# 提问和原文用词不同，怎样找到相关内容？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "lexical_recall_at_1": 0.0,
  "semantic_recall_at_1": 1.0,
  "model_load_seconds": 0.094,
  "index_seconds": 0.031,
  "semantic_query_seconds": 0.016,
  "lexical_query_seconds": 0.0,
  "language": "English controlled comparison",
  "elapsed_seconds": 11.612
}


## 1. 补全：余弦相似度
计算 [3,4] 与 [4,3] 的规范化点积，核对相同向量相似度。

In [2]:
a = np.array([3.0, 4.0])
b = np.array([4.0, 3.0])
a /= np.linalg.norm(a)
b /= np.linalg.norm(b)
assert np.isclose(a @ a, 1) and np.isclose(a @ b, 0.96)
print(a @ b)

0.96


## 2. 修改：Recall@3
用原分数矩阵计算 Recall@3，不能重新选择问题。

In [3]:
top3 = np.argsort(-semantic_scores, axis=1)[:, :3]
recall3 = float(np.mean([g in row for g, row in zip(gold, top3)]))
assert recall3 >= float(np.mean(semantic_top == gold))
print(recall3)

1.0


## 3. 迁移：排名融合
实现 reciprocal rank fusion，每种方法按 1/(60+排名) 累加；零匹配不贡献票数，避免词典顺序制造证据。

In [4]:
def rrf(a, b, k=60):
    combined = np.zeros_like(a)
    for scores in [a, b]:
        for i, row in enumerate(np.argsort(-scores, axis=1)):
            for rank, j in enumerate(row, 1):
                if scores[i, j] > 0:
                    combined[i, j] += 1 / (k + rank)
    return combined
fused = rrf(lexical_scores, semantic_scores)
assert fused.shape == semantic_scores.shape and np.all(fused >= 0) and np.all(fused.sum(axis=1) > 0)
print('融合 Recall@1', np.mean(fused.argmax(1) == gold))

融合 Recall@1 1.0
